In [ ]:
import subprocess
import sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'torch>=2.6,<3', 'transformers==4.57.1', 'peft==0.18.0',
    'accelerate==1.12.0', 'bitsandbytes==0.48.2',
    'soundfile>=0.12,<1', 'numpy>=1.26,<3', 'safetensors>=0.4',
])

In [ ]:
import os
# Trước khi import torch: dùng đúng một GPU để Trainer không tự DataParallel.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path

DATA_ROOT = '/kaggle/input/datasets/datnguyen16/zaic-data/train+public_test/train'
AUDIO_ROOT_OVERRIDE = None
SPEAKER_MAP_PATH = None
OUTPUT_ROOT = Path('/kaggle/working/phowhisper_lora') if Path('/kaggle/working').is_dir() else Path.cwd() / 'phowhisper_lora'
RUN_NAME = 'phowhisper-large-lora-v1'
RESTORE_RUN_DIR = None
RESUME_FROM_CHECKPOINT = None  # None: tự tìm; hoặc đường dẫn checkpoint trong run được khôi phục.
BASE_MODEL = 'vinai/PhoWhisper-large'
BASE_REVISION = 'main'  # Sẽ resolve commit một lần và lưu lại.
SEED = 42
SPLIT_RATIOS = {'train': 0.8, 'validation': 0.1, 'test': 0.1}
SPLIT_RESTARTS = 24
DEDUPLICATE_AUDIO = True
KIND_RATIO_TOLERANCE = 0.05  # Chênh tối đa 5 điểm % so tỷ lệ corpus, chỉ báo cáo nếu không đạt.
SMOKE_TEST = False
USE_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 1e-4
NUM_EPOCHS = 2
MAX_NEW_TOKENS = 440

RUN_DIR = OUTPUT_ROOT / (RUN_NAME + ('-smoke' if SMOKE_TEST else ''))
if not 0 < MAX_NEW_TOKENS <= 440:
    raise ValueError('MAX_NEW_TOKENS cần trong 1..440 cho baseline Whisper này.')

In [ ]:
import csv
import gc
import hashlib
import json
import math
import random
import shutil
import tempfile
import time
import unicodedata
import wave
import zipfile
from collections import Counter, defaultdict
from contextlib import nullcontext
from datetime import datetime, timezone


def canonical_text(text):
    return ' '.join(unicodedata.normalize('NFC', text).casefold().split())


def metric_text(text):
    text = unicodedata.normalize('NFC', text).lower()
    return ' '.join(''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text).split())


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def find_train(root=None):
    root = Path(root).expanduser() if root else (Path('/kaggle/input') if Path('/kaggle/input').is_dir() else Path.cwd())
    if root.is_file():
        return root.resolve()
    candidates = []
    for path in sorted(root.rglob('train.jsonl')):
        with path.open(encoding='utf-8') as stream:
            line = next((line for line in stream if line.strip()), None)
        if line:
            first = json.loads(line)
            if 'audio' in first and 'audio_text' in first:
                candidates.append(path.resolve())
    if len(candidates) != 1:
        raise ValueError(f'Hãy đặt DATA_ROOT cụ thể; tìm được: {candidates}')
    return candidates[0]


def load_speaker_map(path=None):
    mapping = {}
    if path is None:
        return mapping
    path = Path(path)
    with path.open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.DictReader(stream)) if path.suffix.lower() == '.csv' else [json.loads(line) for line in stream if line.strip()]
    for row in rows:
        speaker = row.get('speaker_id')
        if speaker is None or not str(speaker).strip():
            raise ValueError('speaker_id thiếu trong speaker map.')
        if not row.get('clip_id') and not row.get('audio'):
            raise ValueError('Speaker map cần clip_id hoặc audio cho từng dòng.')
        for field in ('clip_id', 'audio'):
            key = row.get(field)
            if key:
                namespaced = (field, str(key))
                if namespaced in mapping and mapping[namespaced] != str(speaker):
                    raise ValueError(f'Speaker map xung đột: {namespaced}')
                mapping[namespaced] = str(speaker)
    return mapping


def read_clips(source, audio_root, speaker_map=None):
    speaker_map = speaker_map or {}
    clips, sample_ids = [], set()
    audio_root = Path(audio_root).resolve()
    with Path(source).open(encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                continue
            item = json.loads(line)
            sid = item['id']
            if sid in sample_ids:
                raise ValueError(f'ID mẫu trùng: {sid}')
            sample_ids.add(sid)
            turns = [(f'{sid}:current', 'current', None, item)]
            turns += [(f'{sid}:history:{i}', 'history', i, turn)
                      for i, turn in enumerate(item.get('history', [])) if turn.get('role') == 'user']
            for cid, kind, history_index, turn in turns:
                text = turn.get('audio_text')
                if not isinstance(text, str) or not metric_text(text):
                    raise ValueError(f'{cid}: thiếu transcript chuẩn.')
                path = (audio_root / turn['audio']).resolve()
                if not path.is_relative_to(audio_root):
                    raise ValueError(f'{cid}: audio nằm ngoài audio root.')
                with wave.open(str(path), 'rb') as wav:
                    if (wav.getframerate(), wav.getnchannels(), wav.getsampwidth(), wav.getcomptype()) != (16000, 1, 2, 'NONE'):
                        raise ValueError(f'{path}: cần WAV PCM mono 16 kHz/16-bit.')
                    frames = wav.getnframes()
                    duration = frames / 16000
                    if not 0 < duration <= 30:
                        raise ValueError(f'{path}: duration={duration}; không tự cắt audio > 30 giây.')
                    pcm = wav.readframes(frames)
                    if len(pcm) != frames*2:
                        raise ValueError(f'{path}: WAV thiếu dữ liệu.')
                candidates = [value for value in (
                    turn.get('speaker_id'), speaker_map.get(('clip_id', cid)), speaker_map.get(('audio', turn['audio']))
                ) if value is not None and str(value).strip()]
                speakers = {str(value) for value in candidates}
                if len(speakers) > 1:
                    raise ValueError(f'{cid}: speaker metadata xung đột.')
                clips.append({
                    'clip_id': cid, 'id': sid, 'kind': kind, 'history_index': history_index,
                    'audio': turn['audio'], 'audio_text': text,
                    'audio_seconds': duration, 'pcm_sha256': hashlib.sha256(pcm).hexdigest(),
                    'speaker_id': next(iter(speakers)) if speakers else None,
                })
    if not clips:
        raise ValueError('Không có audio.')
    return clips


class UnionFind:
    def __init__(self, count):
        self.parent = list(range(count))
    def find(self, index):
        while self.parent[index] != index:
            self.parent[index] = self.parent[self.parent[index]]
            index = self.parent[index]
        return index
    def union(self, left, right):
        a, b = self.find(left), self.find(right)
        if a != b:
            self.parent[max(a, b)] = min(a, b)


def group_and_deduplicate(clips, deduplicate=True):
    uf = UnionFind(len(clips))
    owners = {}
    hashes = defaultdict(list)
    for i, clip in enumerate(clips):
        keys = [('sample', clip['id']), ('pcm', clip['pcm_sha256'])]
        if clip['speaker_id'] is not None:
            keys.append(('speaker', clip['speaker_id']))
        for key in keys:
            if key in owners:
                uf.union(i, owners[key])
            else:
                owners[key] = i
        hashes[clip['pcm_sha256']].append(i)
    duplicates = []
    dropped = set()
    for pcm_hash, indices in hashes.items():
        if len(indices) < 2:
            continue
        if len({metric_text(clips[i]['audio_text']) for i in indices}) != 1:
            raise ValueError(f'Audio PCM trùng nhưng transcript khác nhau: {[clips[i]["clip_id"] for i in indices]}')
        ordered = sorted(indices, key=lambda i: (clips[i]['kind'] != 'current', clips[i]['clip_id']))
        duplicates.append({'pcm_sha256': pcm_hash, 'kept': clips[ordered[0]]['clip_id'],
                           'duplicates': [clips[i]['clip_id'] for i in ordered[1:]]})
        if deduplicate:
            dropped.update(ordered[1:])
    groups = defaultdict(list)
    all_group_ids = {}
    for i, clip in enumerate(clips):
        group_id = f'group-{uf.find(i):06d}'
        all_group_ids[clip['clip_id']] = group_id
        if i not in dropped:
            groups[group_id].append({**clip, 'group_id': group_id})
    return dict(groups), duplicates, all_group_ids


def balanced_group_split(groups, ratios, seed=42, restarts=24):
    names = list(ratios)
    if set(names) != {'train', 'validation', 'test'} or not math.isclose(sum(ratios.values()), 1.0) or any(v <= 0 for v in ratios.values()):
        raise ValueError('Cần ba tỷ lệ train/validation/test > 0, tổng bằng 1.')
    if len(groups) < 3:
        raise ValueError('Cần ít nhất 3 nhóm độc lập để chia train/validation/test mà không tách speaker/PCM/hội thoại.')
    vectors = {gid: [len(rows), sum(r['kind']=='current' for r in rows), sum(r['kind']=='history' for r in rows), sum(r['audio_seconds'] for r in rows)] for gid, rows in groups.items()}
    totals = [sum(v[d] for v in vectors.values()) for d in range(4)]
    scales = [max(total, 1) for total in totals]
    target = {name: [ratios[name]*v for v in totals] for name in names}
    best = None
    for attempt in range(max(1, restarts)):
        rng = random.Random(seed + attempt)
        jitter = {gid: rng.uniform(.8, 1.2) for gid in groups}
        order = sorted(groups, key=lambda gid: (-vectors[gid][0]*jitter[gid], gid))
        counts = {name: [0.0]*4 for name in names}
        assignment = {}
        for gid in order:
            def delta(name):
                return sum(((counts[name][d]+vectors[gid][d]-target[name][d])/scales[d])**2 - ((counts[name][d]-target[name][d])/scales[d])**2 for d in range(4))
            name = min(names, key=lambda name: (delta(name), names.index(name)))
            assignment[gid] = name
            counts[name] = [counts[name][d]+vectors[gid][d] for d in range(4)]
        # Greedy largest-first can overfill train with history-rich dialogues.
        # Improve by moving WHOLE groups, never breaking speaker/PCM boundaries.
        for _ in range(20):
            changed = False
            move_order = list(groups)
            rng.shuffle(move_order)
            for gid in move_order:
                origin = assignment[gid]
                vec = vectors[gid]
                if counts[origin][0] <= vec[0]:
                    continue
                def move_delta(destination):
                    return sum(
                        ((counts[origin][d]-vec[d]-target[origin][d])/scales[d])**2
                        - ((counts[origin][d]-target[origin][d])/scales[d])**2
                        + ((counts[destination][d]+vec[d]-target[destination][d])/scales[d])**2
                        - ((counts[destination][d]-target[destination][d])/scales[d])**2
                        for d in range(4)
                    )
                candidates = [(move_delta(destination), destination) for destination in names if destination != origin]
                improvement, destination = min(candidates)
                if improvement < -1e-12:
                    assignment[gid] = destination
                    counts[origin] = [counts[origin][d]-vec[d] for d in range(4)]
                    counts[destination] = [counts[destination][d]+vec[d] for d in range(4)]
                    changed = True
            if not changed:
                break
        if any(counts[name][0] == 0 for name in names):
            continue
        cost = sum(((counts[name][d]-target[name][d])/scales[d])**2 for name in names for d in range(4))
        if best is None or cost < best[0]:
            best = cost, assignment
    if best is None:
        raise ValueError('Không tạo được ba split khác rỗng; kiểm tra số speaker/nhóm độc lập.')
    splits = {name: [] for name in names}
    for gid, name in best[1].items():
        splits[name].extend(groups[gid])
    for name in names:
        splits[name].sort(key=lambda row: row['clip_id'])
    return splits


def audit_splits(splits):
    for field in ('clip_id', 'id', 'group_id', 'pcm_sha256', 'speaker_id'):
        ownership = {}
        for split, rows in splits.items():
            for row in rows:
                value = row.get(field)
                if value is None:
                    continue
                if value in ownership and ownership[value] != split:
                    raise ValueError(f'Leakage {field}: {value} xuất hiện ở {ownership[value]} và {split}.')
                ownership[value] = split
    return True


def split_statistics(splits):
    return [{
        'split': name, 'clips': len(rows), 'current': sum(r['kind']=='current' for r in rows),
        'history': sum(r['kind']=='history' for r in rows),
        'current_ratio': sum(r['kind']=='current' for r in rows)/len(rows),
        'hours': sum(r['audio_seconds'] for r in rows)/3600,
        'groups': len({r['group_id'] for r in rows}),
        'known_speakers': len({r['speaker_id'] for r in rows if r['speaker_id'] is not None}),
        'unknown_speaker_clips': sum(r['speaker_id'] is None for r in rows),
    } for name, rows in splits.items()]

In [ ]:
SOURCE = find_train(DATA_ROOT)
AUDIO_ROOT = Path(AUDIO_ROOT_OVERRIDE).expanduser().resolve() if AUDIO_ROOT_OVERRIDE else SOURCE.parent
ALL_CLIPS = read_clips(SOURCE, AUDIO_ROOT, load_speaker_map(SPEAKER_MAP_PATH))
GROUPS, DUPLICATES, ALL_GROUP_IDS = group_and_deduplicate(ALL_CLIPS, DEDUPLICATE_AUDIO)
SPLITS = balanced_group_split(GROUPS, SPLIT_RATIOS, SEED, SPLIT_RESTARTS)
audit_splits(SPLITS)
STATS = split_statistics(SPLITS)
print('Source:', SOURCE)
print('Audio:', len(ALL_CLIPS), '| PCM unique:', len({r['pcm_sha256'] for r in ALL_CLIPS}))
print('Duplicate groups:', len(DUPLICATES))
print(json.dumps(STATS, ensure_ascii=False, indent=2))
if all(r['speaker_id'] is None for r in ALL_CLIPS):
    print('Không có speaker_id: đây là split theo hội thoại/PCM, chưa chứng minh speaker-disjoint.')
current_ratio = sum(r['kind']=='current' for rows in SPLITS.values() for r in rows)/sum(len(rows) for rows in SPLITS.values())
for stat in STATS:
    if abs(stat['current_ratio']-current_ratio) > KIND_RATIO_TOLERANCE:
        print('Tỷ lệ current/history lệch mục tiêu, ưu tiên giữ nhóm:', stat['split'])

manifest = {
    'format': 'phowhisper_lora_group_split_v1', 'seed': SEED, 'ratios': SPLIT_RATIOS,
    'source_sha256': file_sha256(SOURCE), 'deduplicate_pcm': DEDUPLICATE_AUDIO,
    'grouping': ['conversation id', 'known speaker_id', 'exact PCM SHA-256'],
    'evaluation_status': 'internal test previously inspected through train benchmarks',
    'duplicates': DUPLICATES, 'statistics': STATS, 'splits': SPLITS,
}
SPLIT_FINGERPRINT = hashlib.sha256(json.dumps(manifest, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
if RESTORE_RUN_DIR is not None and not RUN_DIR.exists():
    shutil.copytree(Path(RESTORE_RUN_DIR), RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RUN_DIR / 'split_manifest.json'
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text(encoding='utf-8'))
    if previous != manifest:
        raise ValueError('Run cũ dùng split khác. Đổi RUN_NAME để tạo run mới.')
else:
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
for name, rows in SPLITS.items():
    with (RUN_DIR / f'{name}.jsonl').open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False)+'\n')
SMOKE_LIMITS = {'train': 32, 'validation': 8, 'test': 8}
WORK_SPLITS = {name: rows[:SMOKE_LIMITS[name]] for name, rows in SPLITS.items()} if SMOKE_TEST else SPLITS
print('Run:', RUN_DIR, '| Smoke test:', SMOKE_TEST)

In [ ]:
import numpy as np
import soundfile as sf
import torch
import transformers
import peft
from torch.utils.data import Dataset
from transformers import (
    AutoConfig, AutoProcessor, AutoModelForSpeechSeq2Seq, BitsAndBytesConfig,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

if not torch.cuda.is_available():
    raise RuntimeError('Notebook fine-tuning cần CUDA: bật GPU trong Kaggle.')
if torch.cuda.device_count() != 1:
    raise RuntimeError('Hãy restart kernel và chạy cell config trước import torch để chỉ dùng một GPU.')
set_seed(SEED)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
existing_config = RUN_DIR / 'training_config.json'
previous_config = json.loads(existing_config.read_text()) if existing_config.exists() else None
model_config = AutoConfig.from_pretrained(BASE_MODEL, revision=previous_config['resolved_base_revision'] if previous_config else BASE_REVISION)
RESOLVED_REVISION = previous_config['resolved_base_revision'] if previous_config else (getattr(model_config, '_commit_hash', None) or BASE_REVISION)
processor = AutoProcessor.from_pretrained(BASE_MODEL, revision=RESOLVED_REVISION, language='vi', task='transcribe')
processor.tokenizer.set_prefix_tokens(language='vi', task='transcribe')
DECODER_START_TOKEN_ID = model_config.decoder_start_token_id
MAX_LABEL_LENGTH = model_config.max_target_positions
print('GPU:', torch.cuda.get_device_name(0))
print('Compute dtype:', COMPUTE_DTYPE, '| Quantization:', 'NF4 4-bit' if USE_4BIT else 'none')
print('Resolved base commit:', RESOLVED_REVISION)
print('torch:', torch.__version__, '| transformers:', transformers.__version__, '| peft:', peft.__version__)


def read_waveform(relative_audio):
    waveform, sr = sf.read(str(AUDIO_ROOT / relative_audio), dtype='float32')
    if sr != 16000 or waveform.ndim != 1 or not len(waveform) or not np.isfinite(waveform).all():
        raise ValueError(f'Waveform không hợp lệ: {relative_audio}')
    return waveform


class WhisperDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
        self.labels = []
        for row in rows:
            labels = processor.tokenizer(row['audio_text']).input_ids
            if len(labels) > MAX_LABEL_LENGTH:
                raise ValueError(f"{row['clip_id']}: label quá dài: {len(labels)} > {MAX_LABEL_LENGTH}")
            self.labels.append(labels)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        features = processor.feature_extractor(
            read_waveform(self.rows[index]['audio']), sampling_rate=16000,
            return_attention_mask=True,
        )
        return {'input_features': features.input_features[0],
                'attention_mask': features.attention_mask[0], 'labels': self.labels[index]}


class WhisperCollator:
    def __call__(self, features):
        batch = {
            'input_features': torch.tensor(np.stack([f['input_features'] for f in features]), dtype=torch.float32),
            'attention_mask': torch.tensor(np.stack([f['attention_mask'] for f in features]), dtype=torch.long),
        }
        padded = processor.tokenizer.pad([{'input_ids': f['labels']} for f in features], return_tensors='pt')
        labels = padded['input_ids'].masked_fill(padded['attention_mask'].ne(1), -100)
        if (labels[:, 0] == DECODER_START_TOKEN_ID).all().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch


train_dataset = WhisperDataset(WORK_SPLITS['train'])
validation_dataset = WhisperDataset(WORK_SPLITS['validation'])
collator = WhisperCollator()
first_batch = collator([train_dataset[0]])
print({key: tuple(value.shape) for key, value in first_batch.items()})

In [ ]:
def distance(left, right):
    if left == right:
        return 0
    if len(left) < len(right):
        left, right = right, left
    previous = list(range(len(right)+1))
    for i, a in enumerate(left, 1):
        current = [i]
        for j, b in enumerate(right, 1):
            current.append(min(previous[j]+1, current[j-1]+1, previous[j-1]+(a != b)))
        previous = current
    return previous[-1]


def score_pair(reference, prediction):
    ref, pred = metric_text(reference), metric_text(prediction)
    if not ref:
        raise ValueError('Reference rỗng sau chuẩn hóa.')
    words, chars = ref.split(), ref.replace(' ', '')
    we, ce = distance(words, pred.split()), distance(chars, pred.replace(' ', ''))
    return {'word_errors': we, 'reference_words': len(words), 'char_errors': ce, 'reference_chars': len(chars),
            'wer': we/len(words), 'cer': ce/len(chars), 'exact_match': ref == pred}


def corpus_metrics(references, predictions):
    if len(references) != len(predictions) or not references:
        raise ValueError('Reference/prediction phải cùng số lượng, khác rỗng.')
    rows = [score_pair(ref, pred) for ref, pred in zip(references, predictions)]
    return {'wer': sum(r['word_errors'] for r in rows)/sum(r['reference_words'] for r in rows),
            'cer': sum(r['char_errors'] for r in rows)/sum(r['reference_chars'] for r in rows),
            'exact_match': sum(r['exact_match'] for r in rows)/len(rows), 'clips': len(rows)}


def compute_metrics(prediction):
    ids = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
    ids = np.where(ids == -100, processor.tokenizer.pad_token_id, ids)
    labels = np.where(prediction.label_ids == -100, processor.tokenizer.pad_token_id, prediction.label_ids)
    predicted = processor.batch_decode(ids, skip_special_tokens=True)
    references = processor.batch_decode(labels, skip_special_tokens=True)
    return corpus_metrics(references, predicted)

In [ ]:
def load_base_model():
    kwargs = {'revision': RESOLVED_REVISION, 'torch_dtype': COMPUTE_DTYPE,
              'device_map': {'': 0}, 'attn_implementation': 'sdpa'}
    if USE_4BIT:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        )
    model = AutoModelForSpeechSeq2Seq.from_pretrained(BASE_MODEL, **kwargs)
    model.generation_config.language = 'vi'
    model.generation_config.task = 'transcribe'
    model.generation_config.forced_decoder_ids = None
    model.generation_config.do_sample = False
    model.generation_config.num_beams = 1
    model.generation_config.return_dict_in_generate = False
    model.generation_config.max_new_tokens = MAX_NEW_TOKENS
    return model


base_model = load_base_model()
base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
base_model.config.use_cache = False


def enable_conv_gradient(module, inputs, output):
    output.requires_grad_(True)


base_model.model.encoder.conv1.register_forward_hook(enable_conv_gradient)
model = get_peft_model(base_model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'v_proj'], bias='none',
))
model.print_trainable_parameters()
assert any(p.requires_grad for p in model.parameters()), 'Không có tham số trainable.'
assert all('lora_' in name for name, parameter in model.named_parameters() if parameter.requires_grad), 'Base chưa được freeze hoàn toàn.'

# Kiểm tra một forward/backward trước khi train thật, chưa cập nhật optimizer.
model.train()
probe = {key: value.to('cuda:0') for key, value in first_batch.items()}
with torch.autocast('cuda', dtype=COMPUTE_DTYPE):
    probe_loss = model(**probe).loss
if not torch.isfinite(probe_loss):
    raise RuntimeError('Loss không hữu hạn ở probe batch.')
probe_loss.backward()
encoder_gradient = any(p.grad is not None and torch.isfinite(p.grad).all().item() and p.grad.abs().sum().item() > 0
                       for name, p in model.named_parameters() if 'encoder' in name and 'lora_' in name)
if not encoder_gradient:
    raise RuntimeError('Encoder LoRA chưa nhận gradient; kiểm tra gradient checkpointing/hook.')
print('Probe loss:', float(probe_loss.detach()), '| Encoder LoRA gradient:', encoder_gradient)
model.zero_grad(set_to_none=True)
del probe, probe_loss

In [ ]:
TRAINING_CONFIG = {
    'format': 'phowhisper_lora_train_v1', 'base_model': BASE_MODEL,
    'resolved_base_revision': RESOLVED_REVISION, 'split_fingerprint': SPLIT_FINGERPRINT,
    'smoke_test': SMOKE_TEST, 'use_4bit': USE_4BIT, 'compute_dtype': str(COMPUTE_DTYPE),
    'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA, 'lora_dropout': LORA_DROPOUT,
    'target_modules': ['q_proj', 'v_proj'], 'seed': SEED,
    'batch_size': TRAIN_BATCH_SIZE, 'gradient_accumulation': GRADIENT_ACCUMULATION_STEPS,
    'learning_rate': LEARNING_RATE, 'epochs': NUM_EPOCHS, 'max_new_tokens': MAX_NEW_TOKENS,
}
if previous_config is not None and previous_config != TRAINING_CONFIG:
    raise ValueError('Training config khác run cũ. Đổi RUN_NAME để tránh trộn checkpoint.')
(RUN_DIR / 'training_config.json').write_text(json.dumps(TRAINING_CONFIG, indent=2)+'\n', encoding='utf-8')
checkpoint_dir = RUN_DIR / 'checkpoints'
resume = RESUME_FROM_CHECKPOINT or (get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None)
if resume is not None:
    resume_path = Path(resume).resolve()
    if not resume_path.is_dir() or not resume_path.is_relative_to(checkpoint_dir.resolve()):
        raise ValueError('Checkpoint phải thuộc checkpoints/ của run đã kiểm tra fingerprint. Dùng RESTORE_RUN_DIR để khôi phục run cũ trước.')
args = Seq2SeqTrainingArguments(
    output_dir=str(checkpoint_dir),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE, warmup_ratio=0.05, weight_decay=0.01,
    num_train_epochs=NUM_EPOCHS, max_steps=3 if SMOKE_TEST else -1,
    fp16=COMPUTE_DTYPE == torch.float16, bf16=COMPUTE_DTYPE == torch.bfloat16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='wer', greater_is_better=False,
    predict_with_generate=True, generation_max_length=448,
    logging_steps=1 if SMOKE_TEST else 10,
    dataloader_num_workers=0, remove_unused_columns=False, label_names=['labels'],
    optim='adamw_torch', report_to=[], seed=SEED, data_seed=SEED,
    eval_accumulation_steps=1,
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_dataset, eval_dataset=validation_dataset,
    data_collator=collator, compute_metrics=compute_metrics, processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
train_result = trainer.train(resume_from_checkpoint=resume)
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation WER:', trainer.state.best_metric)
ADAPTER_DIR = RUN_DIR / 'adapter'
trainer.model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
processor.save_pretrained(ADAPTER_DIR)
trainer.save_state()
TRAIN_STATE = {
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_wer': trainer.state.best_metric,
    'train_metrics': train_result.metrics, 'global_step': trainer.state.global_step,
    'evaluation_status': manifest['evaluation_status'], 'smoke_test': SMOKE_TEST,
    'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__,
    'gpu': torch.cuda.get_device_name(0),
}
(RUN_DIR / 'training_result.json').write_text(json.dumps(TRAIN_STATE, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
print('Adapter:', ADAPTER_DIR)

In [ ]:
del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()
base_model = load_base_model()
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True


def generate_text(current_model, relative_audio):
    inputs = processor.feature_extractor(
        read_waveform(relative_audio), sampling_rate=16000,
        return_tensors='pt', return_attention_mask=True,
    )
    inputs = {key: value.to('cuda:0', dtype=COMPUTE_DTYPE if value.is_floating_point() else value.dtype)
              for key, value in inputs.items()}
    with torch.inference_mode(), torch.autocast('cuda', dtype=COMPUTE_DTYPE):
        generated = current_model.generate(
            **inputs, language='vi', task='transcribe', num_beams=1, do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS, return_timestamps=False,
            return_dict_in_generate=True, use_cache=True,
        ).sequences
    if generated[0, -1].item() != processor.tokenizer.eos_token_id:
        raise RuntimeError('Chạm giới hạn token trước EOS, cần kiểm tra decoding.')
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()


# Preview chỉ dùng validation để không xem test trước khi chạy cell test cuối.
preview = WORK_SPLITS['validation'][0]
print('Chuẩn:', preview['audio_text'])
print('LoRA: ', generate_text(model, preview['audio']))

In [ ]:
def adapter_sha256(directory):
    parts = [(path.name, file_sha256(path)) for path in sorted(Path(directory).glob('adapter*')) if path.is_file()]
    if not parts:
        raise ValueError('Chưa có adapter đã lưu.')
    return hashlib.sha256(json.dumps(parts).encode()).hexdigest()


def evaluate_test(label, disable_adapter=False):
    rows = WORK_SPLITS['test']
    output = RUN_DIR / f'phowhisper-large-{label}.jsonl'
    config = {
        'base_model': BASE_MODEL, 'revision': RESOLVED_REVISION,
        'source_sha256': manifest['source_sha256'], 'split_fingerprint': SPLIT_FINGERPRINT,
        'adapter_sha256': adapter_sha256(ADAPTER_DIR), 'label': label,
        'use_4bit': USE_4BIT, 'dtype': str(COMPUTE_DTYPE),
        'max_new_tokens': MAX_NEW_TOKENS, 'language': 'vi', 'num_beams': 1,
        'smoke_test': SMOKE_TEST,
    }
    config_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    expected = {r['clip_id']: r for r in rows}
    completed = {}
    if output.exists():
        with output.open(encoding='utf-8') as stream:
            for line in stream:
                if not line.strip():
                    continue
                result = json.loads(line)
                cid = result['clip_id']
                if result.get('config_id') != config_id or cid not in expected or cid in completed:
                    raise ValueError(f'File test cũ khác config hoặc ID không hợp lệ: {output}')
                if result.get('reference_text') != expected[cid]['audio_text'] or not isinstance(result.get('asr_text'), str):
                    raise ValueError(f'File test cũ có transcript không hợp lệ: {cid}')
                completed[cid] = result
    pending = [r for r in rows if r['clip_id'] not in completed]
    if pending:
        # A complete final JSON object without newline must not join with the next row.
        if output.exists() and output.stat().st_size:
            with output.open('rb') as stream:
                stream.seek(-1, 2)
                needs_newline = stream.read(1) != b'\n'
            if needs_newline:
                with output.open('a', encoding='utf-8') as stream:
                    stream.write('\n')
        with model.disable_adapter() if disable_adapter else nullcontext():
            generate_text(model, pending[0]['audio'])
            torch.cuda.synchronize()
            with output.open('a', encoding='utf-8') as stream:
                for index, row in enumerate(pending, 1):
                    torch.cuda.synchronize()
                    start = time.perf_counter()
                    text = generate_text(model, row['audio'])
                    torch.cuda.synchronize()
                    seconds = time.perf_counter()-start
                    result = {
                        'clip_id': row['clip_id'], 'id': row['id'], 'kind': row['kind'],
                        'audio': row['audio'], 'reference_text': row['audio_text'], 'asr_text': text,
                        'audio_seconds': row['audio_seconds'], 'inference_seconds': seconds,
                        'model': BASE_MODEL + (' + LoRA' if not disable_adapter else ' (base only)'),
                        'config_id': config_id, 'config': config,
                        **score_pair(row['audio_text'], text),
                    }
                    stream.write(json.dumps(result, ensure_ascii=False)+'\n')
                    stream.flush(); os.fsync(stream.fileno())
                    completed[row['clip_id']] = result
                    if index == 1 or index % 25 == 0 or index == len(pending):
                        print(label, index, '/', len(pending), flush=True)
    return [completed[r['clip_id']] for r in rows]


BEFORE = evaluate_test('before', disable_adapter=True)
AFTER = evaluate_test('after', disable_adapter=False)
assert [r['clip_id'] for r in BEFORE] == [r['clip_id'] for r in AFTER]
COMPARISON = []
for label, results in [('before', BEFORE), ('after', AFTER)]:
    for kind in ('overall', 'current', 'history'):
        subset = [r for r in results if kind == 'overall' or r['kind'] == kind]
        if subset:
            metrics = corpus_metrics([r['reference_text'] for r in subset], [r['asr_text'] for r in subset])
            COMPARISON.append({'model': label, 'group': kind, **metrics})
print(json.dumps(COMPARISON, ensure_ascii=False, indent=2))
with (RUN_DIR / 'before_after.csv').open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(COMPARISON[0]))
    writer.writeheader(); writer.writerows(COMPARISON)
(RUN_DIR / 'before_after.json').write_text(json.dumps({
    'evaluation_status': manifest['evaluation_status'], 'smoke_test': SMOKE_TEST,
    'baseline_precision': 'NF4 4-bit' if USE_4BIT else str(COMPUTE_DTYPE),
    'results': COMPARISON,
}, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')

In [ ]:
from html import escape
from IPython.display import HTML, FileLink, display


def show_table(rows, fields):
    header = ''.join(f'<th>{escape(str(field))}</th>' for field in fields)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(row.get(field, "")))}</td>' for field in fields) + '</tr>' for row in rows)
    display(HTML(f'<div style="overflow-x:auto"><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'))


show_table([{'Model': r['model'], 'Group': r['group'], 'Clips': r['clips'],
             'WER': f"{r['wer']:.2%}", 'CER': f"{r['cer']:.2%}", 'Exact': f"{r['exact_match']:.2%}"}
            for r in COMPARISON], ['Model', 'Group', 'Clips', 'WER', 'CER', 'Exact'])
DIFFERENCES = [{'clip_id': a['clip_id'], 'reference': a['reference_text'], 'before': a['asr_text'], 'after': b['asr_text'],
                'wer_before': a['wer'], 'wer_after': b['wer'], 'delta_wer': b['wer']-a['wer']}
               for a, b in zip(BEFORE, AFTER)]
for reverse, title in ((False, 'Cải thiện nhiều nhất'), (True, 'Kém hơn nhiều nhất')):
    print(title)
    show_table(sorted(DIFFERENCES, key=lambda r: r['delta_wer'], reverse=reverse)[:15],
               ['clip_id', 'reference', 'before', 'after', 'delta_wer'])
with (RUN_DIR / 'per_clip_comparison.csv').open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(DIFFERENCES[0]))
    writer.writeheader(); writer.writerows(DIFFERENCES)

In [ ]:
package = OUTPUT_ROOT / f'{RUN_DIR.name}-adapter-results.zip'
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file() and 'checkpoints' not in path.relative_to(RUN_DIR).parts:
            bundle.write(path, arcname=str(path.relative_to(RUN_DIR)))
print('Adapter:', ADAPTER_DIR)
print('Báo cáo:', RUN_DIR)
print('ZIP:', package)
display(FileLink(os.path.relpath(package, Path.cwd())))

ZIP_FULL_RUN = False  # True nếu muốn đóng gói cả optimizer checkpoints để resume ở phiên mới.
if ZIP_FULL_RUN:
    full_package = shutil.make_archive(str(OUTPUT_ROOT / (RUN_DIR.name+'-resume')), 'zip', root_dir=RUN_DIR)
    print('ZIP resume:', full_package)
    display(FileLink(os.path.relpath(full_package, Path.cwd())))